In [1]:
import pandas as pd


In [2]:
df_data = pd.read_csv(filepath_or_buffer='survey_results_public.csv', low_memory=False)
df_description = pd.read_csv(filepath_or_buffer='survey_results_schema.csv')

print (df_data.shape)
print(df_data.head())


(49191, 172)
   ResponseId                      MainBranch              Age  \
0           1  I am a developer by profession  25-34 years old   
1           2  I am a developer by profession  25-34 years old   
2           3  I am a developer by profession  35-44 years old   
3           4  I am a developer by profession  35-44 years old   
4           5  I am a developer by profession  35-44 years old   

                                           EdLevel  \
0  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   
1              Associate degree (A.A., A.S., etc.)   
2     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   
3     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   
4  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   

                                          Employment  \
0                                           Employed   
1                                           Employed   
2  Independent contractor, freelancer, or self-em...   
3                                        

### Завдання 1. Підрахунок загальної кількості респондентів


In [3]:
responents = df_data['ResponseId'].nunique() #кількість унікальних значень
print ('Загальна кількість респондентів в опитуванні: ', responents)

Загальна кількість респондентів в опитуванні:  49191


### Завдання 2. Аналіз повноти відповідей респондентів


In [4]:
print(df_description['qname']) #список питань
questions = list (set(df_description['qname']).intersection(df_data.columns))
#set(df_description['qname']) - set(df_data.columns) #питання зі списку у файлі опису, які відсутні у файлі результатів
df_data_cleared = df_data[questions].dropna(axis=0, inplace=False) #підрахунок результатів без пропусків

print ('Кількість респондентів, які відповіли на всі запитання з опитування: ', df_data_cleared.shape[0])

0        TechEndorse_1
1        TechEndorse_2
2        TechEndorse_3
3        TechEndorse_4
4        TechEndorse_5
            ...       
134    AIAgentObsWrite
135    AIAgentExternal
136    AIAgentExtWrite
137            AIHuman
138             AIOpen
Name: qname, Length: 139, dtype: str
Кількість респондентів, які відповіли на всі запитання з опитування:  0


### Завдання 3. Статистичний аналіз досвіду респондентів



In [5]:
df_data.WorkExp.describe() #для перевірки

count    42893.000000
mean        13.367403
std         10.800117
min          1.000000
25%          5.000000
50%         10.000000
75%         20.000000
max        100.000000
Name: WorkExp, dtype: float64

In [6]:
pd.DataFrame([{
    'mean': round(df_data['WorkExp'].mean(), 2),
    'median': df_data['WorkExp'].median(),
    'mode': df_data['WorkExp'].mode()[0]
}])

,mean,median,mode
0,13.37,10.0,10.0


### Завдання 4. Аналіз віддаленої роботи

In [7]:
print(df_data.columns[df_data.apply(lambda col: col.astype(str).str.contains('remote', case=False, na=False)).any()]) #пошук колонки з форматом роботи
df_data['RemoteWork'].unique() #перевірка правильності написання варіантів

Index(['RemoteWork', 'TechOppose_15_TEXT', 'JobSatPoints_15_TEXT',
       'DatabaseHaveEntry', 'AIExplain', 'AIOpen'],
      dtype='str')


<StringArray>
[                                                                      'Remote',
                          'Hybrid (some in-person, leans heavy to flexibility)',
                                                                            nan,
                                                                    'In-person',
                               'Hybrid (some remote, leans heavy to in-person)',
 'Your choice (very flexible, you can come in when you want or just as needed)']
Length: 6, dtype: str

In [8]:
df_data_remote = df_data[df_data['RemoteWork']=='Remote']
print(df_data_remote)
print ('Кількість респондентів, які працюють віддалено: ' ,df_data_remote['ResponseId'].nunique() )

       ResponseId                      MainBranch              Age  \
0               1  I am a developer by profession  25-34 years old   
3               4  I am a developer by profession  35-44 years old   
7               8  I am a developer by profession  35-44 years old   
8               9  I am a developer by profession  25-34 years old   
9              10  I am a developer by profession  25-34 years old   
...           ...                             ...              ...   
49137       49138  I am a developer by profession  25-34 years old   
49149       49150  I am a developer by profession  25-34 years old   
49175       49176  I am a developer by profession  25-34 years old   
49179       49180  I am a developer by profession  25-34 years old   
49180       49181  I am a developer by profession  18-24 years old   

                                                 EdLevel Employment  \
0        Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   Employed   
3           Bache

### Завдання 5. Визначення популярності Python

In [9]:
df_data.columns[df_data.isin(['Python']).any()] #колонки з мовами програмування

Index(['TechEndorse_13_TEXT', 'TechOppose_15_TEXT', 'LanguageHaveWorkedWith',
       'LanguageWantToWorkWith', 'LanguageAdmired', 'LanguagesHaveEntry',
       'LanguagesWantEntry', 'SOTagsHaveEntry', 'SOTagsWant Entry',
       'AIAgentKnowWrite', 'AIOpen'],
      dtype='str')

In [10]:
df_python = df_data [df_data['LanguageHaveWorkedWith'].str.contains('Python', na=False)] #створення таблиці з Python-спеціалістами
df_python.to_csv('python_data.csv', index=False)

In [11]:
works_with_python = df_data['LanguageHaveWorkedWith'].str.contains('Python', na=False).sum()
python_percent = (works_with_python / responents)*100
print ('Відсоток респондентів, які програмують на Python: ', round(python_percent, 2),'%')

Відсоток респондентів, які програмують на Python:  37.54 %


### Завдання 6. Аналіз шляхів навчання програмуванню


In [12]:
cols = [c for c in df_data.columns if df_data[c].astype(str).str.contains('Online Courses', na=False).any()]
cols # стовпчики з інформацією про способи навчання

['LearnCode', 'AILearnHow']

In [13]:
print ('Кількість респондентів, які навчалися програмувати через онлайн курси: ', df_data[df_data['LearnCode'].str.contains('Online Courses', na=False)].shape[0])

Кількість респондентів, які навчалися програмувати через онлайн курси:  10973


### Завдання 7. Географічний аналіз компенсації Python-розробників

In [14]:
df_data.columns.isin(['Country']).any() #перевірка чи є колонка за назвою

np.True_

In [15]:
df_python_countries = df_python.dropna(subset=['ConvertedCompYearly'])
result = df_python.dropna(subset=['ConvertedCompYearly']).groupby('Country')[['ConvertedCompYearly']].agg(['mean', 'median'])

df_python_countries.to_csv('df_python_countries.csv', index=False)
pd.DataFrame(result).sort_values(by=('ConvertedCompYearly', 'mean'), ascending=False).round()

ConvertedCompYearly          
                                        mean    median
Country                                               
Oman                                390135.0  390135.0
Andorra                             226104.0  226104.0
Viet Nam                            218837.0    8254.0
United States of America            173299.0  150000.0
Switzerland                         156457.0  142592.0
...                                      ...       ...
Botswana                              1277.0    1277.0
Cambodia                              1270.0    1270.0
Togo                                   354.0     354.0
Palestine                               78.0      78.0
Antigua and Barbuda                      1.0       1.0

[153 rows x 2 columns]

### Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів


In [16]:
most_paid = df_data.dropna(subset=['ConvertedCompYearly']).sort_values(by='ConvertedCompYearly', ascending=False)[:5]
pd.DataFrame(most_paid['EdLevel'])

,EdLevel
34267,"Associate degree (A.A., A.S., etc.)"
28700,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
43143,"Associate degree (A.A., A.S., etc.)"
35353,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)"
45971,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"


### Завдання 9. Аналіз популярності Python по віковим категоріям


In [17]:
result = (df_python.groupby('Age')['ResponseId'].count() / df_data.groupby('Age')['ResponseId'].count() * 100).round(2)
pd.DataFrame(result).rename(columns={"ResponseId": "Response_python_percantage, %"})

,"Response_python_percantage, %"
Age,
18-24 years old,40.00
25-34 years old,36.94
35-44 years old,36.72
45-54 years old,38.63
55-64 years old,37.24
65 years or older,31.63
Prefer not to say,31.22


### Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників


In [18]:
df_data.columns.isin(['Industry']).any() #перевірка чи є колонка за назвою
#df_data['Industry'].unique() #перевірка варіантів індустрії

np.True_

In [19]:
percentile_75 = df_data['ConvertedCompYearly'].quantile(0.75)
df_most_paid = df_data_remote[df_data_remote['ConvertedCompYearly']>=percentile_75]
industry_most_paid = df_most_paid['Industry'].value_counts()
pd.DataFrame(industry_most_paid)

,count
Industry,
Software Development,1186
Fintech,190
Healthcare,188
Other:,176
"Internet, Telecomm or Information Services",138
Banking/Financial Services,88
Government,78
Media & Advertising Services,75
Retail and Consumer Services,65
